# Introduktion til Clock Glitching

**RESUMÉ:** *Mikrokontrollere og FPGA'er har en række driftsbetingelser, der skal overholdes, for at enheden fungerer korrekt. Uden for disse betingelser begynder enheder at fejlfungere, og mere ekstreme overtrædelser kan få enheden til helt at stoppe eller endda blive beskadiget. Ved at gå uden for disse driftsbetingelser i meget korte tidsrum kan vi fremkalde en række midlertidige fejlfunktioner.*

*I denne øvelse udforsker vi clock glitching, som indsætter korte glitches i en enheds clock. Dette vil blive brugt til at få et mål, der summerer tal i en løkke, til at nå et forkert resultat.*

**LÆRINGSMÅL:**

* Forstå effekterne af clock glitching
* Udforske ChipWhisperer's glitch-modul
* Bruge clock glitching til at forstyrre et måls algoritme

## Teori om Clock Glitching

Digitale hardwareenheder forventer næsten altid en form for pålidelig clock. Vi kan manipulere den clock, der præsenteres for enheden, for at forårsage utilsigtet adfærd. Vi fokuserer her på mikrokontrollere, men andre digitale enheder (f.eks. hardware-krypteringsacceleratorer) kan også påføres fejl ved hjælp af denne teknik.

Lad os starte med en mikrokontroller. Følgende figur er et uddrag fra Atmel AVR ATMega328P databladet:

![A2_1](img/Mcu-unglitched.png)

I stedet for at indlæse hver instruktion fra FLASH og udføre hele eksekveringen har systemet en pipeline til at fremskynde eksekveringsprocessen. Det betyder, at en instruktion dekodes, mens den næste hentes, som vist i følgende diagram:

![A2_2](img/Clock-normal.png)

Men hvis vi ændrer klokken, kan vi få en situation, hvor systemet ikke har nok tid til rent faktisk at udføre en instruktion. Betragt følgende, hvor Execute #1 reelt springes over. Inden systemet har tid til at udføre den, kommer en ny clockkant, hvilket får mikrokontrolleren til at begynde eksekveringen af den næste instruktion:

![A2_3](img/Clock-glitched.png)

Dette får mikrokontrolleren til at springe en instruktion over. Sådanne angreb kan i praksis være enormt kraftfulde. Betragt for eksempel følgende kode fra `linux-util-2.24`:

```C
/*
 *   auth.c -- PAM authorization code, common between chsh and chfn
 *   (c) 2012 by Cody Maloney <cmaloney@theoreticalchaos.com>
 *
 *   this program is free software.  you can redistribute it and
 *   modify it under the terms of the gnu general public license.
 *   there is no warranty.
 *
 */

#include "auth.h"
#include "pamfail.h"

int auth_pam(const char *service_name, uid_t uid, const char *username)
{
    if (uid != 0) {
        pam_handle_t *pamh = NULL;
        struct pam_conv conv = { misc_conv, NULL };
        int retcode;

        retcode = pam_start(service_name, username, &conv, &pamh);
        if (pam_fail_check(pamh, retcode))
            return FALSE;

        retcode = pam_authenticate(pamh, 0);
        if (pam_fail_check(pamh, retcode))
            return FALSE;

        retcode = pam_acct_mgmt(pamh, 0);
        if (retcode == PAM_NEW_AUTHTOK_REQD)
            retcode =
                pam_chauthtok(pamh, PAM_CHANGE_EXPIRED_AUTHTOK);
        if (pam_fail_check(pamh, retcode))
            return FALSE;

        retcode = pam_setcred(pamh, 0);
        if (pam_fail_check(pamh, retcode))
            return FALSE;

        pam_end(pamh, 0);
        /* no need to establish a session; this isn't a
         * session-oriented activity...  */
    }
    return TRUE;
}
```

Dette er login-koden for Linux-operativsystemet. Bemærk, at hvis vi kunne springe tjekket af `if (uid != 0)` over og blot hoppe til slutningen, kunne vi undgå at skulle indtaste en adgangskode. Det er styrken ved glitch-angreb - ikke at vi bryder kryptering, men simpelthen omgår hele autentificeringsmodulet!

### Glitch-hardware

ChipWhisperer Glitch-systemet bruger den samme synkrone metode som Side Channel Analysis (SCA) capture. Et systemclock (som enten kan komme fra ChipWhisperer eller Device Under Test (DUT)) bruges til at generere glitches. Disse glitches indsættes derefter tilbage i klokken, selvom det er muligt at bruge glitchene alene til andre formål (f.eks. til spændingsglitching, EM-glitching).

Genereringen af glitches sker med to variable faseskiftsmoduler, konfigureret som følger:

![A2_4](img/Glitchgen-phaseshift.png)

I CW-Husky er der én vigtig forskel: Phase Shift 1-udgangen inverteres ikke, inden den AND-kobles med Phase Shift 2-udgangen.

Enable-linjen bruges til at bestemme, hvornår glitches indsættes. Glitches kan indsættes kontinuerligt (nyttigt til udvikling) eller udløses af en hændelse. Følgende figur viser, hvordan glitchen kan multiplexes til output til Device Under Test (DUT).

![A2_5](img/Glitchgen-mux.png)

### Hardwaresupport: CW-Husky

Clock-genereringslogikken i Husky's 7-serie FPGA er betydeligt anderledes end 6-serie FPGA'erne brugt i CW-Lite/Pro. DCM er erstattet af den langt mere kraftfulde (og strømkrævende...) Mixed Mode Clock Manager (MMCM). Især for vores glitching-applikation giver MMCM'er fine faseskiftjusteringer over et ubegrænset område i trin så små som 15ps. Og alt dette uden at skulle dynamisk rekonfigurere FPGA-bitfilen! Af denne grund er formatet for angivelse af glitch-offset og bredde forskelligt fra hvad det var for CW-Lite/Pro. I stedet for at angive en procentdel af kildeclock-perioden angiver du nu det faktiske antal faseskiftstrin. Varigheden af et faseskifttrin er 1/56 af MMCM VCO-clockperioden, som selv kan konfigureres til at være et sted i området fra 600 MHz til 1200 MHz (via `scope.clock.pll.update_fpga_vco()`).

Selvom MMCM er mere kraftfuld end DCM med hensyn til funktioner, kræver den også meget mere strøm. Af denne grund er glitch-genereringskredsen deaktiveret som standard og skal eksplicit tændes. Frygt ikke - Husky bruger også Xilinx's XADC-modul til kontinuerligt at overvåge temperaturen, og alle MMCM'er slukkes automatisk, når temperaturen begynder at blive for høj, langt inden farlige niveauer nås (kør `scope.XADC` for at se alle statistikker og indstillinger).

In [ ]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWHUSKY'
SS_VER = 'SS_VER_2_1'

In [ ]:
%run "../jupyter/Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../firmware/mcu/simpleserial-glitch
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

In [ ]:
fw_path = "../firmware/mcu/simpleserial-glitch/simpleserial-glitch-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)

Vi vil sandsynligvis crashe målet et par gange, mens vi prøver noget glitching. Lav en funktion til at genstarte målet:

In [ ]:
def reboot_flush():            
    scope.io.nrst = False
    time.sleep(0.05)
    scope.io.nrst = "high_z"
    time.sleep(0.05)
    #Flush garbage too
    target.flush()

## Kommunikation

I denne øvelse introducerer vi en ny metode: `target.simpleserial_read_witherrors()`. Vi forventer et simpleserial-svar tilbage; men en glitch vil ofte få målet til at crashe og returnere en ugyldig streng. Denne metode håndterer alt det for os. Den fortæller os også, om svaret var gyldigt, og hvad fejlreturkoden var. Bruges som følger:

In [ ]:
reboot_flush()
target.simpleserial_write("g", bytearray([]))

val = target.simpleserial_read_witherrors('r', 4, glitch_timeout=10)#For loop check
valid = val['valid']
if valid:
    response = val['payload']
    raw_serial = val['full_response']
    error_code = val['rv']

print(val)

## Target Firmware

For denne øvelse er vores mål at få følgende kode til at producere et forkert resultat:

```C
uint8_t glitch_loop(uint8_t* in)
{
    volatile uint16_t i, j;
    volatile uint32_t cnt;
    cnt = 0;
    trigger_high();
    for(i=0; i<50; i++){
        for(j=0; j<50; j++){
            cnt++;
        }
    }
    trigger_low();
    simpleserial_put('r', 4, (uint8_t*)&cnt);
    return (cnt != 2500);
}
```

Som du kan se, har vi en simpel løkke. Dette er et rigtig godt sted at starte glitching af to grunde:

1. Vi har en rigtig lang del af tiden med mange instruktioner at glitche. I modsætning hertil forsøger vi med Linux-eksemplet at ramme en enkelt instruktion.

1. For nogle glitching-scenarier leder vi efter en ret specifik glitch-effekt. I Linux-eksemplet er vi muligvis afhængige af, at glitchen får målet til at springe en instruktion over frem for at korruptere sammenligningen, da det er langt mere sandsynligt at bringe os derhen i kodeforløbet. For denne simple løkkeberegning vil næsten enhver fejlfunktion vise sig i resultatet.

## Glitch-modul

Alle indstillinger/metoder for glitch-modulet kan tilgås under `scope.glitch`. Som sædvanlig kan dokumentation for indstillinger og metoder tilgås på [ReadtheDocs](https://chipwhisperer.readthedocs.io/en/latest/api.html) eller med Python-kommandoen `help`:

In [ ]:
help(scope.glitch)

#### Vi gennemgår først indstillinger, der er forskellige mellem CW Husky og CW Lite/Pro:
* clk_src

> Det clocksignal, som glitch-DCM bruger som input. Kan indstilles til "target" eller "clkgen". I dette tilfælde leverer vi clock til målet, så vi ønsker dette indstillet til "clkgen".

> På CW Husky bruges en separat PLL til at clocke glitch-modulet i stedet for clkgen-modulet. Den tilsvarende indstilling her for "clkgen" er "pll"
* offset

> Hvor i output-clocket glitchen skal placeres. Kan være i området `[-48.8, 48.8]`. Ofte vil vi prøve mange offsets, når vi forsøger at glitche et mål.

> På CW Husky vil området afhænge af frekvensen af PLL'en, der driver glitch-modulet (kan konfigureres til et sted i området fra 600 MHz til 1200 MHz via `scope.clock.pll.update_fpga_vco()`), men når glitch-modulet er aktivt, vil området være `[0, scope.glitch.phase_shift_steps]`.
* width

> Hvor bred glitchen skal være. Kan være i området `[-50, 50]`, selvom der ikke er nogen grund til at bruge bredder < 0. Bredere glitches forårsager lettere glitches, men er også mere tilbøjelige til at crashe målet, hvilket betyder, at vi ofte ønsker at prøve en række bredder, når vi angriber et mål.

> Ligesom offset vil området være `[0, scope.glitch.phase_shift_steps]`.

#### Disse indstillinger er derimod de samme mellem Husky og Lite/Pro:

* output

> Det output, som glitch-modulet producerer. Til clock-glitching er clock_xor ofte den mest nyttige mulighed, da dette inverterer clocket under glitchen.
* ext_offset

> Antallet af clockcykler efter triggeren, hvor glitchen placeres.
* repeat

> Antallet af clockcykler, som glitchen gentages for. Højere værdier øger antallet af instruktioner, der kan glitches, men øger ofte risikoen for at crashe målet.

* trigger_src

> Hvordan glitchen udløses. I dette laboratorium ønsker vi automatisk at udløse glitchen fra triggerpinnen, men kun efter at have armeret ChipWhisperer'en, så vi bruger `ext_single`

Derudover skal vi fortælle ChipWhisperer at bruge glitch-modulets output som clockkilde for målet ved at indstille `scope.io.hs2 = "glitch"`. Vi opsætter også en stor `repeat` for at gøre glitching lettere.

## CW Glitch Controller

For at gøre det lettere at oprette en glitch-løkke inkluderer ChipWhisperer en glitch controller. Vi starter med at initialisere den med forskellige mulige resultater af angrebet. Du definerer disse til hvad du vil, men ofte er tre grupper tilstrækkelige:

1. `"success"`, hvor vores glitch havde den ønskede effekt
1. `"reset"`, hvor vores glitch havde en uønsket effekt. Ofte er denne effekt at crashe eller genstarte målet, og det er derfor vi kalder det `"reset"`
1. `"normal"`, hvor din glitch ikke havde en mærkbar effekt.

Vi skal også fortælle den, hvilke glitch-parametre vi vil scanne igennem, i dette tilfælde width og offset:

In [ ]:
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset"])

En af fordelene ved glitch controlleren er, at den kan vise vores aktuelle indstillinger. Dette opdateres i realtid, mens vi bruger glitch controlleren!

In [ ]:
gc.display_stats()

Vi kan også lave et indstillingsplot, der også kan opdatere i realtid:

In [ ]:
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None})

Her er `plotdots` et Python dictionary, der specificerer, hvordan du vil plotte hver gruppe. I dette tilfælde plotter vi `"success"` som et grønt `+` (`"+g"`), `"reset"` som et rødt x (`"xr"`), og vi plotter ikke glitch-forsøg, hvor intet unormalt sker (`None`).

Dette plot opdaterer automatisk sine grænser, efterhånden som punkter tilføjes. Hvis du vil angive aksegrænserne, kan du gøre det som følger:

```python
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_bound=(-48, 48), y_bound=(-48, 48))
```

Du kan også vælge, hvilke parametre du vil bruge til x og y, enten ved indeks eller ved navn:

```python
# will flip width and offset axes
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_index=1, y_index=0)
# or
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, x_index="offset", y_index="width")

```

Du kan indstille intervaller for hver glitch-indstilling:

In [ ]:
gc.set_range("width", -5, 5)
gc.set_range("offset", -5, 5)

Hver indstilling bevæger sig fra min til max baseret på det globale trin:

In [ ]:
gc.set_global_step([5.0, 2.5])

Vi kan udskrive alle glitch-indstillinger for at se, hvordan dette ser ud:

In [ ]:
for glitch_setting in gc.glitch_values():
    print("offset: {:4.1f}; width: {:4.1f}".format(glitch_setting[1], glitch_setting[0]))

Du kan fortælle glitch controlleren, hvornår du har nået en bestemt resultattilstand, som følger:

In [ ]:
#gc.add("reset", (scope.glitch.width, scope.glitch.offset)) or simply gc.add("reset")
#gc.add("success", (scope.glitch.width, scope.glitch.offset)) or simply gc.add("success")

Fra og med ChipWhisperer 5.7 kan du springe parametrene for glitch-bredde og glitch-offset over. I dette tilfælde bruger glitch controlleren sine interne værdier for koordinaterne. Bemærk, at på grund af afrunding vil dette normalt være lidt anderledes end den faktiske hardwareværdi på Lite/Pro; dog vil værdierne stadig svare til de korrekte indstillinger på din ChipWhisperer.

For CW-Husky skal vi først eksplicit tænde for glitch-kredsløbet (det er slukket som standard for at spare strøm):

In [ ]:
if scope._is_husky:
    scope.glitch.enabled = True

Vi starter med følgende indstillinger. Det er normalt bedst at bruge "clock_xor" til clock-glitching, som vil indsætte en glitch, hvis clocket er højt eller clocket er lavt.

In [ ]:
#Basic setup
# set glitch clock
if scope._is_husky:
    scope.glitch.clk_src = "pll"
else:
    scope.glitch.clk_src = "clkgen" 

scope.glitch.output = "clock_xor" # glitch_out = clk ^ glitch
scope.glitch.trigger_src = "ext_single" # glitch only after scope.arm() called

scope.io.hs2 = "glitch"  # output glitch_out on the clock line
print(scope.glitch)

Disse indstillinger er ofte et godt udgangspunkt for al clock-glitching, så, nyt i ChipWhisperer 5.7, har vi en metode, der opsætter alt dette for dig:

In [ ]:
#scope.cglitch_setup()

Du bør have alt, hvad du behøver, til at konstruere din glitch-løkke. Vi hjælper dig i gang, men resten er op til dig! Desuden er der noget at huske på:

* Du skal detektere crashes, vellykkede glitches og normale returneringer fra målet. Vær ikke bange for at eksperimentere med løkken: du kan altid genstarte den og køre koden igen.
* Du kan dække et større sæt glitch-indstillinger ved at starte med store glitch controller-trin for at få en idé om, hvor nogle interessante steder er, og derefter gentage glitch-løkken med små trin i interessante områder. Hvor der er én vellykket glitch, er der sandsynligvis flere!
* Du kan fremskynde din glitch-kampagne markant ved kun at plotte crashes og succeser, da de typisk er meget sjældnere end normal adfærd i målet.
* På CW-Husky angives glitch-offset og bredde i antal faseskiftstrin, hvorimod på CW-Lite/Pro angives de i procentdel af clockperioden. Koden nedenfor sætter passende startintervaller for hvert tilfælde. Kør `help(scope.glitch)` for at forstå dette bedre.

In [ ]:
import struct

# Disable logging
cw.set_all_log_levels(cw.logging.CRITICAL)

# Sætter forsinkelsen for glitch-signalet til 2 clock-cykler efter den eksterne trigger
scope.glitch.ext_offset = 2

# width and offset numbers have a very different meaning for Husky vs Lite/Pro;
# see help(scope.glitch) for details
if scope._is_husky:
    # Sætter søgeområdet for glitch-bredden fra 0 til det maksimale antal fase-skift trin
    gc.set_range("width", 0, scope.glitch.phase_shift_steps)

    # Sætter søgeområdet for glitch-offset fra 0 til det maksimale antal fase-skift trin
    gc.set_range("offset", 0, scope.glitch.phase_shift_steps)

    # Sætter det globale søgeskrift til først 200, derefter 100 for at søge groft og derefter fint
    gc.set_global_step([200, 100])

    # Husky overvåger ADC-samples on-the-fly under en capture og vil markere fejl, hvis
    # der opstår klipning (scope.gain.db er for høj), eller hvis ikke nok af det dynamiske 
    # område bruges (scope.gain.db er for lav). Disse kontroller er aktiveret som standard,
    # men de kan deaktiveres ved at kalde:
    scope.adc.lo_gain_errors_disabled = True
    scope.adc.clip_errors_disabled = True
else:
    gc.set_range("width", 0, 48)
    gc.set_range("offset", -48, 48)
    gc.set_global_step([8, 4, 2, 1, 0.4])

# Sætter antallet af gentagne glitch-pulser til 10 for hver glitch-hændelse
scope.glitch.repeat = 10

# Sætter timeout for ADC (analog-til-digital konverter) til 0.1 sekunder
scope.adc.timeout = 0.1

# Genstarter target og tømmer bufferen for gammel data
reboot_flush()

for glitch_setting in gc.glitch_values():
    
    scope.glitch.offset = glitch_setting[1]
    scope.glitch.width = glitch_setting[0]

    if scope.adc.state:
        # Trigger still high!
        print("h", end="")
        gc.add("reset")

        # Genstarter target og tømmer bufferen for gammel data
        reboot_flush()

    # Klargør og armerer scope til at vente på en trigger-hændelse
    scope.arm()

    # Sender kommandoen "g" til target via SimpleSerial protokollen med en tom data-pakke
    target.simpleserial_write("g", bytearray([]))

    # Optager data fra scope og gemmer resultatet i variablen ret
    ret = scope.capture()
    
    if ret:
        # Timeout - no trigger
        print("n", end="")
        gc.add("reset")

        # Genstarter target og tømmer bufferen for gammel data
        reboot_flush()
    else:
        # Læser 4 bytes fra target via SimpleSerial protokollen med en glitch-timeout på 10ms og en 
        # generel timeout på 50ms, og gemmer resultatet inklusiv eventuelle fejl i variablen val
        val = target.simpleserial_read_witherrors('r', 4, glitch_timeout=10, timeout=50)
    
        if val['valid'] is False:
            gc.add("reset")
        else:
            if val['payload'] is None:
                print(val['payload'])
                continue
                
            # gcnt is the loop counter
            gcnt = struct.unpack("<I", val['payload'])[0]
            
            if gcnt != 2500: 
                gc.add("success")
                print("")
                print(val['payload'])
                print(gcnt)                
                print("🐙", end="")
                break
            else:
                gc.add("normal")

### Plotting af Glitch-resultater

Vi kan genplotte vores glitch-kort ved hjælp af metoden `plot_2d()`. Indstillinger svarer til `glitch_plot()`. Hvis `plotdots` ikke er angivet, bruges de samme som i `glitch_plot()`.

In [ ]:
gc.plot_2d(plotdots={"success":"+g", "reset":"xr", "normal":None})

Sørg for at notere disse glitch-indstillinger, da vi vil bruge dem i resten af glitching-laboratorierne! Faktisk bruger vi meget af den generelle kodestruktur her i resten af laboratorierne, hvor de eneste store ændringer er:

### Repeat

Denne øvelse brugte en ret stor repeat-værdi. Som navnet antyder, styrer denne indstilling, hvor mange gange glitchen gentages (dvs. en repeat-værdi på 5 placerer glitches i 5 på hinanden følgende clockcykler). Tænk på, at hver indsatte glitch har en chance for enten at forårsage en glitch eller crashe enheden. Dette var ret fordelagtigt for denne øvelse, da vi havde mange forskellige steder, vi ønskede at placere en glitch - brug af en høj repeat-værdi øgede vores chance for et crash, men øgede også vores chance for en vellykket glitch. For et angreb, hvor vi retter mod en enkelt instruktion, øger vi faktisk ikke vores glitch-chance overhovedet, men har stadig den øgede crashrisiko. Endnu værre kan en vellykket glitch på det forkerte sted også forårsage et crash! Det er af denne grund, at det ofte er bedre at bruge en lav repeat-værdi, når man retter mod en enkelt instruktion.

### Ext Offset

Ext offset-indstillingen styrer en forsinkelse mellem triggerens aktivering og glitchens indsættelse. Ligesom repeat er det baseret på hele clockcykler, hvilket betyder, at et ext offset på 10 indsætter en glitch 10 cykler efter triggeren aktiveres. Vi behøvede ikke bekymre os om denne indstilling for denne øvelse, da den store repeat-værdi var i stand til at tage os ind i det område, vi ønskede. Dette vil ikke være tilfældet for mange applikationer, hvor du bliver nødt til at prøve glitches ved et stort udvalg af ext_offsets.

### Success, Reset og Normal

Disse tre resultattilstande er normalt nok til at beskrive de fleste glitch-resultater. Hvad der udgør en succes vil dog ændre sig baseret på, hvilken firmware du angriber. For eksempel, hvis vi angreb Linux-autentificeringen, ville vi måske basere succes på et tjek for at se, om vi er root eller ej.

In [ ]:
scope.dis()
target.dis()